# Olist Source Relationship & Cardinality Profiling

This notebook is used to validate the relationships between the Olist source tables before dimensional warehouse design.

The checks focus on:

- Table cardinality
- Foreign-key integrity
- Composite key uniqueness
- One-to-one and one-to-many relationships
- Customer identity behavior
- Potential row-multiplication risks

The results from this notebook will be documented in `docs/relationship_cardinality_profile.md`.

## 1. Load Source Data

Upload the Olist source CSV files into the Colab session, then load them into pandas DataFrames for relationship and integrity testing.

In [1]:
import pandas as pd
from google.colab import files

uploaded = files.upload()

orders = pd.read_csv("olist_orders_dataset.csv")
order_items = pd.read_csv("olist_order_items_dataset.csv")
payments = pd.read_csv("olist_order_payments_dataset.csv")
reviews = pd.read_csv("olist_order_reviews_dataset.csv")
customers = pd.read_csv("olist_customers_dataset.csv")
products = pd.read_csv("olist_products_dataset.csv")
sellers = pd.read_csv("olist_sellers_dataset.csv")
translations = pd.read_csv("product_category_name_translation.csv")

Saving olist_customers_dataset.csv to olist_customers_dataset.csv
Saving olist_geolocation_dataset.csv to olist_geolocation_dataset.csv
Saving olist_order_items_dataset.csv to olist_order_items_dataset.csv
Saving olist_order_payments_dataset.csv to olist_order_payments_dataset.csv
Saving olist_order_reviews_dataset.csv to olist_order_reviews_dataset.csv
Saving olist_orders_dataset.csv to olist_orders_dataset.csv
Saving olist_products_dataset.csv to olist_products_dataset.csv
Saving olist_sellers_dataset.csv to olist_sellers_dataset.csv
Saving product_category_name_translation.csv to product_category_name_translation.csv


## 2. Orders → Order Items

This section validates the relationship between orders and individual order-item records.

The analysis checks:

- Whether every order item matches a valid order
- How many orders contain multiple items
- Average and maximum item count per order
- Whether the composite key `order_id + order_item_id` is unique

This relationship is expected to be **one-to-many**.

In [2]:
# Count order-item records per order
item_counts = order_items.groupby("order_id").size()

print("Orders with item records:", item_counts.shape[0])
print("Orders with multiple items:", (item_counts > 1).sum())
print("Average items per order:", round(item_counts.mean(), 2))
print("Maximum items in one order:", item_counts.max())

# Check for orders with no item records
orders_without_items = ~orders["order_id"].isin(order_items["order_id"])
print("Orders without item records:", orders_without_items.sum())

# Check for orphaned order-item records
orphan_items = ~order_items["order_id"].isin(orders["order_id"])
print("Order-item rows without matching order:", orphan_items.sum())

# Check composite key uniqueness
duplicate_item_keys = order_items.duplicated(
    subset=["order_id", "order_item_id"]
).sum()

print("Duplicate order_id + order_item_id combinations:", duplicate_item_keys)

Orders with item records: 98666
Orders with multiple items: 9803
Average items per order: 1.14
Maximum items in one order: 21
Orders without item records: 775
Order-item rows without matching order: 0
Duplicate order_id + order_item_id combinations: 0


### Findings

- 98,666 orders contain at least one item record.
- 9,803 orders contain multiple items.
- Orders contain an average of 1.14 items, with a maximum of 21 items in a single order.
- 775 orders have no corresponding item records.
- No orphaned order-item records were found.
- No duplicate `order_id + order_item_id` combinations were found.

These results confirm that the relationship between orders and order items is **one-to-many**, and that `order_id + order_item_id` is a valid composite key candidate for the order-items table.

## 3. Orders → Payments

This section evaluates the relationship between orders and payment records.

The analysis checks:

- Whether every payment record matches a valid order
- How many orders contain multiple payment records
- Maximum number of payment records associated with one order
- Whether the composite key `order_id + payment_sequential` is unique

Because an order may contain multiple payment records, this relationship is expected to be **one-to-many**.

In [3]:
# Count payment records per order
payment_counts = payments.groupby("order_id").size()

print("Orders with payment records:", payment_counts.shape[0])
print("Orders with multiple payments:", (payment_counts > 1).sum())
print("Maximum payment records for one order:", payment_counts.max())

# Check for orders with no payment records
orders_without_payments = ~orders["order_id"].isin(payments["order_id"])
print("Orders without payment records:", orders_without_payments.sum())

# Check for orphaned payment records
orphan_payments = ~payments["order_id"].isin(orders["order_id"])
print("Payment rows without matching order:", orphan_payments.sum())

# Check composite key uniqueness
duplicate_payment_keys = payments.duplicated(
    subset=["order_id", "payment_sequential"]
).sum()

print("Duplicate order_id + payment_sequential combinations:", duplicate_payment_keys)

Orders with payment records: 99440
Orders with multiple payments: 2961
Maximum payment records for one order: 29
Orders without payment records: 1
Payment rows without matching order: 0
Duplicate order_id + payment_sequential combinations: 0


### Findings

- 99,440 orders contain at least one payment record.
- 2,961 orders contain multiple payment records.
- The maximum number of payment records associated with a single order is 29.
- Only 1 order has no corresponding payment record.
- No orphaned payment records were found.
- No duplicate `order_id + payment_sequential` combinations were found.

These results confirm that the relationship between orders and payments is **one-to-many**, and that `order_id + payment_sequential` is a valid composite key candidate for the payments table.

Because some orders contain multiple payment records, payments should remain at their own grain to avoid row multiplication when combined with order-item data.

## 4. Orders → Reviews

This section examines the relationship between orders and customer review records.

The analysis checks:

- Whether every review matches a valid order
- How many orders have no review
- How many orders contain multiple review records
- Whether `review_id` is unique
- Whether the same `review_id` can be associated with multiple orders

This validation is important because the review table may not follow a strict one-review-per-order structure.

In [4]:
# Count review records per order
review_counts = reviews.groupby("order_id").size()

print("Orders with review records:", review_counts.shape[0])
print("Orders without review records:",
      (~orders["order_id"].isin(reviews["order_id"])).sum())

print("Orders with multiple reviews:", (review_counts > 1).sum())
print("Maximum reviews for one order:", review_counts.max())

# Check for orphaned review records
orphan_reviews = ~reviews["order_id"].isin(orders["order_id"])
print("Review rows without matching order:", orphan_reviews.sum())

# Check whether review_id is unique
print("Is review_id unique?:", reviews["review_id"].is_unique)

# Count duplicate review_id values
duplicate_review_ids = reviews["review_id"].duplicated().sum()
print("Duplicate review_id occurrences:", duplicate_review_ids)

# Check whether the same review_id is associated with multiple orders
review_order_counts = reviews.groupby("review_id")["order_id"].nunique()

print(
    "review_id values linked to multiple orders:",
    (review_order_counts > 1).sum()
)

Orders with review records: 98673
Orders without review records: 768
Orders with multiple reviews: 547
Maximum reviews for one order: 3
Review rows without matching order: 0
Is review_id unique?: False
Duplicate review_id occurrences: 814
review_id values linked to multiple orders: 789


### Findings

- 98,673 orders contain at least one review record.
- 768 orders have no corresponding review record.
- 547 orders contain multiple review records.
- The maximum number of reviews associated with a single order is 3.
- No orphaned review records were found.
- `review_id` is not unique.
- 814 duplicate `review_id` occurrences were identified.
- 789 `review_id` values are associated with more than one order.

These results show that the reviews table does **not** follow a strict one-review-per-order or one-row-per-review-ID structure.

Because `review_id` is not unique, it should not be treated as a standalone primary key without further investigation. The reviews table should likely remain at its own grain, and review metrics may need to be aggregated before being joined to other order-level or item-level facts.

## 5. Customers → Orders

This section validates the relationship between customer records and orders using `customer_id`.

The analysis checks:

- Whether every order has a matching customer record
- Whether `customer_id` is unique in the customers table
- Whether one `customer_id` appears in multiple orders

At the `customer_id` level, this relationship is expected to behave as **one-to-one**.

In [5]:
# Check whether customer_id is unique in the customers table
print(
    "Is customer_id unique in customers?:",
    customers["customer_id"].is_unique
)

# Check for orders without a matching customer
orphan_orders = ~orders["customer_id"].isin(customers["customer_id"])
print("Orders without matching customer:", orphan_orders.sum())

# Count orders per customer_id
customer_order_counts = orders.groupby("customer_id").size()

print(
    "customer_id values associated with multiple orders:",
    (customer_order_counts > 1).sum()
)

print(
    "Maximum orders associated with one customer_id:",
    customer_order_counts.max()
)

Is customer_id unique in customers?: True
Orders without matching customer: 0
customer_id values associated with multiple orders: 0
Maximum orders associated with one customer_id: 1


### Findings

- `customer_id` is unique in the customers table.
- Every order has a matching customer record.
- No `customer_id` values are associated with multiple orders.
- The maximum number of orders associated with a single `customer_id` is 1.

These results confirm that `customer_id` behaves as a **one-to-one relationship** between the customers and orders tables.

However, this does not mean each real customer appears only once in the dataset. Repeat purchasing must be analyzed using `customer_unique_id`, which is evaluated in the next section.

## 6. Customer Unique ID Analysis

Olist provides both `customer_id` and `customer_unique_id`.

`customer_id` identifies the customer record associated with an individual order, while `customer_unique_id` is intended to identify the same underlying customer across multiple purchases.

This section checks:

- Number of unique customers
- Number of customers associated with multiple `customer_id` values
- Maximum number of customer records linked to one unique customer
- Percentage of customers represented by only one customer record

This distinction is important for repeat-purchase, purchase-frequency, and customer lifetime analysis.

In [6]:
# Count customer_id records associated with each customer_unique_id
unique_customer_counts = (
    customers.groupby("customer_unique_id")
    .size()
)

print(
    "Unique customers:",
    customers["customer_unique_id"].nunique()
)

print(
    "Customers with multiple customer_id records:",
    (unique_customer_counts > 1).sum()
)

print(
    "Maximum customer_id records for one unique customer:",
    unique_customer_counts.max()
)

print(
    "Customers represented only once:",
    (unique_customer_counts == 1).sum()
)

print(
    "Percent of customers represented only once:",
    round((unique_customer_counts == 1).mean() * 100, 2),
    "%"
)

Unique customers: 96096
Customers with multiple customer_id records: 2997
Maximum customer_id records for one unique customer: 17
Customers represented only once: 93099
Percent of customers represented only once: 96.88 %


### Findings

- The dataset contains 96,096 unique customers based on `customer_unique_id`.
- 2,997 unique customers are associated with multiple `customer_id` records.
- The maximum number of `customer_id` records linked to a single unique customer is 17.
- 93,099 customers appear only once.
- 96.88% of unique customers are represented by only one `customer_id`.

These results confirm that `customer_unique_id` should be used for repeat-purchase, purchase-frequency, and customer lifetime analysis.

Using `customer_id` alone would incorrectly treat repeat purchases from the same underlying customer as separate customers.

## 7. Products → Order Items

This section validates the relationship between products and transactional order-item records.

The analysis checks:

- Whether `product_id` is unique in the products table
- Whether every product referenced in order items has a matching product record
- How many unique products appear in transactions
- Whether any products in the products table never appear in order-item data

This relationship is expected to be **one-to-many**.

In [7]:
# Check whether product_id is unique
print(
    "Is product_id unique in products?:",
    products["product_id"].is_unique
)

# Check for order-item rows without a matching product
orphan_product_rows = ~order_items["product_id"].isin(products["product_id"])
print(
    "Order-item rows without matching product:",
    orphan_product_rows.sum()
)

# Count unique products appearing in transactions
products_in_orders = order_items["product_id"].nunique()
print(
    "Unique products appearing in order items:",
    products_in_orders
)

# Check for products that never appear in order-item data
products_never_ordered = ~products["product_id"].isin(order_items["product_id"])
print(
    "Products never appearing in order items:",
    products_never_ordered.sum()
)

# Count order-item rows per product
product_item_counts = order_items.groupby("product_id").size()

print(
    "Products appearing in multiple order-item rows:",
    (product_item_counts > 1).sum()
)

print(
    "Maximum order-item rows for one product:",
    product_item_counts.max()
)

Is product_id unique in products?: True
Order-item rows without matching product: 0
Unique products appearing in order items: 32951
Products never appearing in order items: 0
Products appearing in multiple order-item rows: 14834
Maximum order-item rows for one product: 527


### Findings

- `product_id` is unique in the products table.
- Every product referenced in the order-items table has a matching product record.
- All 32,951 products in the products table appear in transactional order-item data.
- 14,834 products appear in multiple order-item rows.
- The most frequently occurring product appears in 527 order-item rows.
- No orphaned product references were found.

These results confirm a **one-to-many** relationship between products and order items.

Because every product appears in transactional data and `product_id` is unique in the products table, the products table is a strong candidate for a product dimension in the future warehouse.

## 8. Sellers → Order Items

This section validates the relationship between sellers and transactional order-item records.

The analysis checks:

- Whether `seller_id` is unique in the sellers table
- Whether every seller referenced in order items has a matching seller record
- How many unique sellers appear in transactions
- How many sellers are associated with multiple orders
- Maximum number of orders associated with a single seller

This relationship is expected to be **one-to-many**.

In [8]:
# Check whether seller_id is unique
print(
    "Is seller_id unique in sellers?:",
    sellers["seller_id"].is_unique
)

# Check for order-item rows without a matching seller
orphan_seller_rows = ~order_items["seller_id"].isin(sellers["seller_id"])
print(
    "Order-item rows without matching seller:",
    orphan_seller_rows.sum()
)

# Count unique sellers appearing in transactions
sellers_in_orders = order_items["seller_id"].nunique()
print(
    "Unique sellers appearing in order items:",
    sellers_in_orders
)

# Count distinct orders per seller
seller_order_counts = (
    order_items.groupby("seller_id")["order_id"]
    .nunique()
)

print(
    "Sellers associated with multiple orders:",
    (seller_order_counts > 1).sum()
)

print(
    "Maximum orders associated with one seller:",
    seller_order_counts.max()
)

# Check for sellers that never appear in order-item data
sellers_without_sales = ~sellers["seller_id"].isin(order_items["seller_id"])
print(
    "Sellers never appearing in order items:",
    sellers_without_sales.sum()
)

Is seller_id unique in sellers?: True
Order-item rows without matching seller: 0
Unique sellers appearing in order items: 3095
Sellers associated with multiple orders: 2524
Maximum orders associated with one seller: 1854
Sellers never appearing in order items: 0


### Findings

- `seller_id` is unique in the sellers table.
- Every seller referenced in the order-items table has a matching seller record.
- All 3,095 sellers appear in transactional order-item data.
- 2,524 sellers are associated with multiple orders.
- The most active seller is associated with 1,854 distinct orders.
- No sellers exist in the source table without corresponding order-item activity.
- No orphaned seller references were found.

These results confirm a **one-to-many** relationship between sellers and order items.

Because `seller_id` is unique in the sellers table and every seller participates in transactional activity, the sellers table is a strong candidate for a seller dimension in the future warehouse.

## 9. Product Categories → English Translation

The products table stores category names in Portuguese, while a separate translation table provides English category names.

This section checks:

- How many products are missing a category value
- How many distinct categories appear in the products table
- How many categories are available in the translation table
- Whether any populated product categories lack an English translation
- Whether any translation-table categories are unused

These checks will help determine how product categories should be standardized before warehouse loading.

In [9]:
# Count products with missing category values
missing_product_categories = products["product_category_name"].isna().sum()
print(
    "Products missing category:",
    missing_product_categories
)

# Work only with products that have a populated category
products_with_category = products[
    products["product_category_name"].notna()
]

# Count distinct categories in products
product_category_count = products_with_category[
    "product_category_name"
].nunique()

print(
    "Distinct categories in products:",
    product_category_count
)

# Count distinct categories in translation table
translation_category_count = translations[
    "product_category_name"
].nunique()

print(
    "Categories in translation table:",
    translation_category_count
)

# Find product rows whose category has no translation
missing_translation_mask = ~products_with_category[
    "product_category_name"
].isin(
    translations["product_category_name"]
)

print(
    "Products with category but no English translation:",
    missing_translation_mask.sum()
)

# Count distinct untranslated category names
untranslated_categories = products_with_category.loc[
    missing_translation_mask,
    "product_category_name"
].unique()

print(
    "Distinct product categories without translation:",
    len(untranslated_categories)
)

print(
    "Untranslated category names:",
    untranslated_categories
)

# Check for translation categories that are not used by any product
unused_translation_categories = ~translations[
    "product_category_name"
].isin(
    products_with_category["product_category_name"]
)

print(
    "Translation categories unused by products:",
    unused_translation_categories.sum()
)

Products missing category: 610
Distinct categories in products: 73
Categories in translation table: 71
Products with category but no English translation: 13
Distinct product categories without translation: 2
Untranslated category names: ['pc_gamer' 'portateis_cozinha_e_preparadores_de_alimentos']
Translation categories unused by products: 0


### Findings

- 610 products are missing a product category.
- 73 distinct category names appear in the products table.
- The translation table contains 71 category translations.
- 13 products use category values that do not have an English translation.
- Those 13 products belong to 2 untranslated categories:
  - `pc_gamer`
  - `portateis_cozinha_e_preparadores_de_alimentos`
- All categories in the translation table are used by at least one product.

These results show that the category translation table provides nearly complete coverage, but two valid product categories lack English translations.

The cleaning layer should preserve these categories and assign standardized English labels manually or through a documented fallback rule rather than dropping the affected products.

## 10. Foreign-Key Integrity Summary

This section performs a final set of foreign-key checks across the major source-table relationships.

The goal is to confirm whether transactional records reference valid parent records before dimensional modeling begins.

The relationships tested are:

- `orders.customer_id` → `customers.customer_id`
- `order_items.order_id` → `orders.order_id`
- `order_items.product_id` → `products.product_id`
- `order_items.seller_id` → `sellers.seller_id`
- `payments.order_id` → `orders.order_id`
- `reviews.order_id` → `orders.order_id`

A value of zero indicates that no orphaned child records were found for that relationship.

In [10]:
# Foreign-key integrity checks

fk_checks = {
    "Orders without matching customer": (
        ~orders["customer_id"].isin(customers["customer_id"])
    ).sum(),

    "Order items without matching order": (
        ~order_items["order_id"].isin(orders["order_id"])
    ).sum(),

    "Order items without matching product": (
        ~order_items["product_id"].isin(products["product_id"])
    ).sum(),

    "Order items without matching seller": (
        ~order_items["seller_id"].isin(sellers["seller_id"])
    ).sum(),

    "Payments without matching order": (
        ~payments["order_id"].isin(orders["order_id"])
    ).sum(),

    "Reviews without matching order": (
        ~reviews["order_id"].isin(orders["order_id"])
    ).sum()
}

for check, result in fk_checks.items():
    print(f"{check}: {result}")

Orders without matching customer: 0
Order items without matching order: 0
Order items without matching product: 0
Order items without matching seller: 0
Payments without matching order: 0
Reviews without matching order: 0


### Findings

- Every order references a valid customer.
- Every order-item record references a valid order.
- Every order-item record references a valid product.
- Every order-item record references a valid seller.
- Every payment record references a valid order.
- Every review record references a valid order.
- No orphaned records were found across the major source-table relationships.

These results indicate strong referential integrity across the core Olist source tables.

This gives us confidence that future warehouse transformations can rely on these relationships without needing to repair missing parent-child references.

## 11. Profiling Summary

The relationship and cardinality profiling confirmed that the Olist source data is structurally reliable, while also identifying several important modeling considerations.

### Key Findings

- Orders → order items is one-to-many.
- Orders → payments is one-to-many.
- Orders → reviews is mostly one-to-one, but some orders contain multiple review records.
- `review_id` is not unique and should not be treated as a standalone primary key.
- `customer_id` behaves one-to-one with orders, while `customer_unique_id` is required for identifying repeat customers.
- Products and sellers both have clean one-to-many relationships with order items.
- Two product categories lack English translations.
- No orphaned foreign-key records were found across the major relationships.
- Different source-table grains create a risk of row multiplication if tables are joined without aggregation.

### Modeling Implication

The future dimensional warehouse should preserve separate fact-table grains for orders, order items, payments, and reviews.

Measures from one-to-many tables should be aggregated appropriately before being combined in reporting or analytical queries.

These findings will now be used to update `docs/relationship_cardinality_profile.md` and guide dimensional warehouse design.